In [ ]:
!pip install streamlit -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 43.4 MB/s eta 0:00:00


In [ ]:
!pip install faiss-cpu groq transformers peft torch -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 52.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 10.1 MB/s eta 0:00:00


In [ ]:
%%writefile /content/backend.py
import torch
import faiss
import json
import numpy as np
from transformers import CLIPModel, CLIPProcessor
from groq import Groq

device = "cuda" if torch.cuda.is_available() else "cpu"
model_512 = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
processor_512 = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

client = Groq(api_key="YOUR_GROQ_API_KEY")

index_top = faiss.read_index("/content/top_db.index")
index_bottom = faiss.read_index("/content/bottom_db.index")
index_dress = faiss.read_index("/content/dress_db.index")
index_outer = faiss.read_index("/content/outer_db.index")

with open("/content/top_paths.json") as f:
    top_paths = json.load(f)
with open("/content/bottom_paths.json") as f:
    bottom_paths = json.load(f)
with open("/content/dress_paths.json") as f:
    dress_paths = json.load(f)
with open("/content/outer_paths.json") as f:
    outer_paths = json.load(f)

def extract_fashion_keywords(query):
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": f"""
너는 패션 전문가야. CLIP 검색용 영어 키워드 5~10개 추출해줘.
입력: "{query}"
출력: 영어 키워드만 콤마로 구분. 다른 말 하지 마.
"""}]
    )
    return [k.strip() for k in response.choices[0].message.content.strip().split(",")]

def get_text_embedding(query_text):
    inputs = processor_512(text=[query_text], return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        embedding = model_512.get_text_features(**inputs)
    embedding = embedding / embedding.norm(dim=-1, keepdim=True)
    return embedding.cpu().numpy()

def search_category(query_text, index, paths, top_k=5):
    if index.ntotal == 0:
        return []
    top_k = min(top_k, index.ntotal)
    text_vec = get_text_embedding(query_text)
    scores, indices = index.search(text_vec, top_k)
    return [{"path": paths[idx], "score": float(score)} for score, idx in zip(scores[0], indices[0])]

def get_outfit_combinations(query, sel_top, sel_bottom, sel_dress):
    keywords = extract_fashion_keywords(query)
    text_query = ", ".join(keywords)
    result = {"keywords": keywords, "tops": [], "bottoms": [], "dresses": [], "combinations": []}
    if sel_top:
        result["tops"] = search_category(text_query, index_top, top_paths)
    if sel_bottom:
        result["bottoms"] = search_category(text_query, index_bottom, bottom_paths)
    if sel_dress:
        result["dresses"] = search_category(text_query, index_dress, dress_paths)
    for top in result["tops"]:
        for bottom in result["bottoms"]:
            result["combinations"].append({
                "top": top["path"],
                "bottom": bottom["path"],
                "score": (top["score"] + bottom["score"]) / 2
            })
    result["combinations"].sort(key=lambda x: -x["score"])
    result["combinations"] = result["combinations"][:3]
    return result

Writing /content/backend.py


In [ ]:
%%writefile app.py
import streamlit as st
import time
from backend import get_outfit_combinations

# 1. 페이지 설정
st.set_page_config(page_title="Find Closet", layout="wide")

# 2. 세션 상태 초기화
if "current_page" not in st.session_state:
    st.session_state.current_page = "home"
if "search_query" not in st.session_state:
    st.session_state.search_query = ""
if "sel_top" not in st.session_state:
    st.session_state.sel_top = False
if "sel_bottom" not in st.session_state:
    st.session_state.sel_bottom = False
if "sel_dress" not in st.session_state:
    st.session_state.sel_dress = False

# 3. 사이드바
with st.sidebar:
    st.title("🔍 Find Closet")
    st.write("---")
    if st.button("🏠 홈 (스타일 추천)", use_container_width=True):
        st.session_state.current_page = "home"
        st.rerun()
    if st.button("👗 내 옷장 관리", use_container_width=True):
        st.session_state.current_page = "closet"
        st.rerun()
    st.write("---")
    st.info("🤖 **AI STATUS**\n\nCLIP Model Ready")

# --- PAGE 1: 홈 ---
if st.session_state.current_page == "home":
    st.title("🔍 파인드 클로젯")
    st.subheader("자신의 옷장을 검색하고 상황에 맞는 최고의 옷을 추천받으세요.")
    st.write("---")

    st.markdown("### ⚙️ 추천받고 싶은 카테고리를 선택하세요")
    col_b1, col_b2, col_b3 = st.columns(3)
    with col_b1:
        top_type = "primary" if st.session_state.sel_top else "secondary"
        if st.button("👕 상의", type=top_type, use_container_width=True):
            st.session_state.sel_top = not st.session_state.sel_top
            st.rerun()
    with col_b2:
        bottom_type = "primary" if st.session_state.sel_bottom else "secondary"
        if st.button("👖 하의", type=bottom_type, use_container_width=True):
            st.session_state.sel_bottom = not st.session_state.sel_bottom
            st.rerun()
    with col_b3:
        dress_type = "primary" if st.session_state.sel_dress else "secondary"
        if st.button("👗 원피스", type=dress_type, use_container_width=True):
            st.session_state.sel_dress = not st.session_state.sel_dress
            st.rerun()

    st.write("<br>", unsafe_allow_html=True)
    query = st.text_input("💡 어떤 스타일을 찾으시나요?", placeholder="예: '결혼식 하객룩 추천해줘'")

    if st.button("추천받기", type="primary", use_container_width=True):
        if not (st.session_state.sel_top or st.session_state.sel_bottom or st.session_state.sel_dress):
            st.error("⚠️ 카테고리 중 최소 하나를 선택해 주세요!")
        elif not query.strip():
            st.warning("⚠️ 스타일 문장을 입력해 주세요.")
        else:
            st.session_state.search_query = query
            st.session_state.current_page = "results"
            st.rerun()

# --- PAGE 2: 결과 ---
elif st.session_state.current_page == "results":
    if st.button("← 메인으로 돌아가기"):
        st.session_state.current_page = "home"
        st.rerun()

    st.title("✨ 스타일 추천 결과")
    st.success(f"**'{st.session_state.search_query}'** 기반으로 내 옷장에서 최적의 조합을 찾았습니다.")

    selected_tabs = []
    selected_categories = []
    if st.session_state.sel_top:
        selected_tabs.append("👕 상의")
        selected_categories.append("Top")
    if st.session_state.sel_bottom:
        selected_tabs.append("👖 하의")
        selected_categories.append("Bottom")
    if st.session_state.sel_dress:
        selected_tabs.append("👗 원피스")
        selected_categories.append("Dress")

    with st.spinner("AI가 코디를 분석 중이에요..."):
        result = get_outfit_combinations(
            st.session_state.search_query,
            st.session_state.sel_top,
            st.session_state.sel_bottom,
            st.session_state.sel_dress
        )

    st.markdown(f"**🏷️ AI 키워드:** {', '.join(result['keywords'])}")
    st.write("---")

    tabs = st.tabs(selected_tabs)
    for idx, cat in enumerate(selected_categories):
        with tabs[idx]:
            if cat == "Top":
                items = result["tops"]
            elif cat == "Bottom":
                items = result["bottoms"]
            else:
                items = result["dresses"]

            if not items:
                st.warning("해당 카테고리에 옷이 없어요! 옷장에 추가해주세요.")
            else:
                cols = st.columns(len(items))
                for rank, item in enumerate(items):
                    with cols[rank]:
                        st.image(item["path"], use_container_width=True)
                        st.markdown(f"### TOP {rank+1}")
                        st.info(f"🔥 유사도: {item['score']:.3f}")

    if result["combinations"]:
        st.write("---")
        st.markdown("### 👗 최적 코디 조합 Top 3")
        for i, combo in enumerate(result["combinations"]):
            st.markdown(f"**{i+1}번 코디** (점수: {combo['score']:.3f})")
            col1, col2 = st.columns(2)
            with col1:
                st.image(combo["top"], caption="상의")
            with col2:
                st.image(combo["bottom"], caption="하의")

# --- PAGE 3: 옷장 관리 ---
elif st.session_state.current_page == "closet":
    st.title("👗 내 옷장 관리")

    if st.button("➕ 옷 추가하기"):
        st.session_state.show_uploader = not st.session_state.get("show_uploader", False)

    if st.session_state.get("show_uploader", False):
        uploaded_file = st.file_uploader("의류 사진 업로드 (JPG, PNG)")
        if uploaded_file:
            with st.spinner("CLIP 모델 속성 분석 중..."):
                time.sleep(1)
            st.success("옷이 추가됐어요!")
            st.session_state.show_uploader = False
            st.rerun()

Writing app.py


In [ ]:
# 1. 스트림릿 백그라운드 실행
!nohup streamlit run app.py >/dev/null 2>&1 &

# 2. 클라우드플레어 터널 다운로드 및 실행
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64
!./cloudflared-linux-amd64 tunnel --url http://localhost:8501

2026-06-08T03:11:32Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-06-08T03:11:32Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-06-08T03:11:36Z INF +--------------------------------------------------------------------------------------------+
2026-06-08T03:11:36Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-06-08T03:11:36Z INF |  https://nevertheless-inexpensive-decor-checked.tryclo